<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/courses/c41-header.png" alt="Nextia Learning · Speech AI: Transcribe, Understand and Speak" width="100%">

# Solution: Speech or silence?

**[Speech or silence?](https://learning.nextia-ai.com/courses/speech/m03/speech-or-silence/)** · Speech AI: Transcribe, Understand and Speak · Module 3, lesson 1 of 3 · [learning.nextia-ai.com](https://learning.nextia-ai.com)

**You will learn to** find the speech in a call with an energy threshold, score it against the true turns (missed speech and false speech), and see where it fails: music on hold and background noise.

| | |
|---|---|
| **Time** | About 30 minutes |
| **Needs** | An internet connection for the setup cells. No account, no key and no model download. |
| **You should know** | Loudness in dBFS and frames, from [Set up the speech project](https://learning.nextia-ai.com/courses/speech/m01/set-up-the-speech-project/). |
| **Tested** | Python 3.12 with NumPy 2.5.4, SciPy 1.18.1, soundfile 0.14.0 and matplotlib 3.11.2 (scikit-learn 1.9.1, jiwer 4.0.0 and faster-whisper 1.2.1 where the lesson needs them), in Jupyter on a laptop CPU (Apple M4, 16 GB), from a fresh start (*Restart and run all*) |

### How to use this notebook

This notebook has every query, task and check of the lesson. The [lesson page](https://learning.nextia-ai.com/courses/speech/m03/speech-or-silence/) has the full explanations, the diagrams and the knowledge checks that count toward your certificate. Keep both open.

1. Run the cells in order, from the top. Press **Shift+Enter** to run a cell and move to the next one.
2. At a **Predict** note, write your prediction down before you run the cell.
3. At a **Your turn** note, write your answer in the cell, run it, then run the check cell below it.
4. If you are stuck, open the [solution notebook](https://github.com/Nextia-Intelligence/nextia-learning-labs/blob/main/C41/M03-L01-speech-or-silence/speech-or-silence-solution.ipynb).

## Setup: install the packages

Run the next cell. It installs the versions that the course uses: numpy 2.5.4, scipy 1.18.1, soundfile 0.14.0, and matplotlib for the pictures. It needs an internet connection and takes about a minute the first time.

You should see one line per package, with the versions above. If you see other versions, restart the kernel (in Colab: **Runtime › Restart session**; in Kaggle: **Run › Restart & clear cell outputs**), then run this cell again.

In [ ]:
%pip install -q --disable-pip-version-check numpy==2.5.4 scipy==1.18.1 soundfile==0.14.0 matplotlib==3.11.2
import warnings; warnings.filterwarnings("ignore")
from importlib.metadata import version
for name in ['numpy', 'scipy', 'soundfile']:
    print(name, version(name))

## Setup: get the Larkfield calls

Run the next cell. It downloads the files that this lesson needs (about 8 MB) into a folder `larkfield-calls`, with the course's helper `compare.py`, and checks each file's checksum. It needs an internet connection and no account, and takes about 20 seconds.

You should see `Ready:` and the number of files. If you run the cell again, it uses the files that are already there.

If the cell shows an error, read its last line:

- `Could not download`: check your internet connection, then run the cell again. In Kaggle, turn on **Internet** in the notebook settings.
- `wrong checksum`: delete the folder `larkfield-calls` (in Colab: the **Files** panel on the left), then run the cell again.

Where the calls come from: Larkfield is an invented shop. Our code wrote the scripts, and open text-to-speech voices (Piper and Kokoro) read them. Every call is **synthetic**. The [dataset card](https://github.com/Nextia-Intelligence/nextia-learning-labs/blob/main/C41/data/dataset-card.md) names each voice and its licence. The calls are free to reuse under CC BY 4.0.

In [ ]:
# Setup: download the files into larkfield-calls/ and check them.
import hashlib, os, shutil, subprocess, urllib.request
from pathlib import Path

LABS = "https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/C41/"
FILES = {
    "data/calls/calls.jsonl": "22fe4e147bf62b4ff6c3b6b4a1cc559b048e12a0c2b3c8098eb72982f2d9e695",
    "data/calls/phone/C-0022.wav": "f1958f7b460c5f040dfd0f84a31cb83677ee66b25b514df21a086d9d16db5a51",
    "data/calls/phone/C-0098.wav": "58d8793dc53dcedc78b0d1f8f9616ea38044d8f76a321c76ab007cbaaa39bfca",
    "data/calls/phone/C-0204.wav": "2fc187d2960de73f32beb554c90781e0a2853520dfce12582d47de0e5f6911e5",
    "data/calls/phone/C-0093.wav": "68125fe62023b1ec52064e488a45ffdcce48fa5c602ece56c2fa16f2a9a2e424",
    "data/calls/phone/C-0181.wav": "6bce67f941673d66ec1a2cf271c4c7872721a7936dbb841ac927bc856ab96726",
    "data/calls/phone/C-0003.wav": "e03c761cb765170c39bcc4d676e3f4f7144f510c38ae476182d4cb5432792761",
    "data/noise/music_on_hold.wav": "ecdc8ca83a89079c2acd2e4d9d93d47fb725650c25643de67f9332645a1771fd",
    "data/noise/background_talk.wav": "232f4c029b07247ca4207d8752e1cdb7784eb35222218176d12470ce23355ab0",
    "data/noise/street.wav": "8474ae365d3231942d439734fc2ac0b8bf499be7382b382de1eaf4d888c299fb",
    "data/noise/line.wav": "560e52c3c810a50803e6e5af0eeaaa7febff384f46519033d9739edaa5c170e4",
    "compare.py": "db394eb044e0423706644ba7c8e6451e4d21a268293b868032918ea2f30db85b",
}
HERE = Path.cwd()
# The notebook works in its own folder, marked with a file, so it never
# writes into your own project if you open the notebook there.
if Path(".nextia-notebook").exists():  # the cell ran before: start from the parent folder
    os.chdir("..")
    HERE = Path.cwd()
PROJECT = Path("larkfield-calls").resolve()
(PROJECT / "data").mkdir(parents=True, exist_ok=True)
(PROJECT / ".nextia-notebook").touch()

def sha256(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()

def download(url, path):
    try:
        urllib.request.urlretrieve(url, path)
    except OSError:
        # Python without certificates (python.org on macOS): try curl.
        if subprocess.run(["curl", "-fsSL", "-o", str(path), url]).returncode:
            raise SystemExit(f"Could not download {url}. Check your internet connection.")

for name, expected in FILES.items():
    path = PROJECT / name
    path.parent.mkdir(parents=True, exist_ok=True)
    if not (path.exists() and sha256(path) == expected):
        local = HERE / name  # a copy next to the notebook, if there is one
        if local.exists():
            shutil.copy(local, path)
        else:
            download(LABS + name, path)
        if sha256(path) != expected:
            raise SystemExit(f"{name} has the wrong checksum. Delete the folder larkfield-calls and run again.")

os.chdir(PROJECT)  # work in the project folder, as in the lesson
print(f"Ready: {len(FILES)} files in larkfield-calls.")

## Setup: helpers

The next cell loads NumPy as `np`, soundfile as `sf`, SciPy's `signal`, matplotlib, the audio player `Audio`, and the course helper as `cp` (normalisation, word error rate and the comparison table). It defines a few small helpers:

- `load_calls()` reads the labels of every call; `reference(call)` joins its turns into the reference transcript.
- `read_audio(path, sr, channel)` reads a recording as numbers between -1 and 1; `dbfs(x)` gives its loudness.
- `true_turns(call)` gives the true (start, end, speaker) of each turn, from the first word to the last word.

It also defines the check helpers. They tell you if your answer is right, without showing the answer:

- `expect(what, yours, expected)` compares one of your numbers with the expected one, and says "too high" or "too low".
- `expect_hash(what, yours, expected)` compares any result with a hidden fingerprint of the expected one.

Run the cell. It prints nothing.

In [ ]:
# Helpers: calls, audio, and checks that do not show the answer.
import hashlib, json, warnings
from math import gcd
import numpy as np
import soundfile as sf
import matplotlib.pyplot as plt
from scipy import signal
from IPython.display import Audio, display
import compare as cp
warnings.filterwarnings("ignore")

def load_calls(path="data/calls/calls.jsonl"):
    """The labels of every call, by call ID."""
    rows = [json.loads(line) for line in open(path, encoding="utf-8") if line.strip()]
    return {c["call_id"]: c for c in rows}

def reference(call, speaker=None):
    """The reference transcript: the text of all turns (or of one speaker's turns), in time order."""
    return " ".join(t["text"] for t in call["turns"] if speaker in (None, t["speaker"]))

def read_audio(path, sr=None, channel=None):
    """(numbers between -1 and 1, samples a second). channel=None mixes the channels; 0 or 1 keeps one side."""
    x, file_sr = sf.read(str(path), dtype="float32", always_2d=True)
    x = x.mean(axis=1) if channel is None else x[:, channel]
    if sr and sr != file_sr:
        g = gcd(file_sr, sr)
        x = signal.resample_poly(x, sr // g, file_sr // g).astype(np.float32)
    return x, sr or file_sr

def dbfs(x):
    """Loudness in decibels below full scale: 0 is the loudest; speech on a phone is often -30 to -15."""
    return float(20 * np.log10(np.sqrt(np.mean(np.square(x))) + 1e-9))

def true_turns(call):
    """The true turns: (start, end, speaker), from the first word to the last word of each turn."""
    return [(t["words"][0]["start"], t["words"][-1]["end"], t["speaker"]) if t.get("words")
            else (t["start"], t["end"], t["speaker"]) for t in call["turns"]]

def fingerprint(value):
    def norm(v):
        if hasattr(v, "tolist"):
            return norm(v.tolist())
        if isinstance(v, dict):
            return {str(k): norm(x) for k, x in sorted(v.items(), key=lambda kv: str(kv[0]))}
        if isinstance(v, (list, tuple)):
            return [norm(x) for x in v]
        if isinstance(v, bool) or v is None or isinstance(v, int):
            return v
        if isinstance(v, float):
            return None if v != v else round(v, 3)
        if hasattr(v, "item"):
            return norm(v.item())
        return str(v)
    return hashlib.sha256(json.dumps(norm(value), sort_keys=True).encode()).hexdigest()[:16]

def expect(what, yours, expected, tolerance=1e-6):
    """Compare one number or value with the expected one."""
    if isinstance(yours, (np.integer, np.floating)):
        yours = yours.item()
    if yours is None or yours is Ellipsis:
        print(f"Not yet: {what} has no value. Write your answer in the cell above, then run it again.")
        return
    if isinstance(expected, float) and isinstance(yours, (int, float)):
        same = abs(yours - expected) <= tolerance * max(1, abs(expected))
    else:
        same = yours == expected
    if same:
        print(f"Check passed: {what} is {yours}.")
    elif isinstance(expected, (int, float)) and isinstance(yours, (int, float)):
        print(f"Not yet: {what} is {yours}, which is too {'high' if yours > expected else 'low'}.")
    else:
        print(f"Not yet: {what} is {yours!r}, which is not the expected value.")

def expect_hash(what, yours, expected):
    """Compare any result with the fingerprint of the expected one."""
    if yours is None or yours is Ellipsis:
        print(f"Not yet: {what} has no value. Write your answer in the cell above, then run it again.")
    elif fingerprint(yours) == expected:
        print(f"Check passed: {what} is right.")
    else:
        print(f"Not yet: {what} is not the expected result. Read the task again, and check each step.")

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

## 1. The idea: loud frames are speech

A recogniser should only get the parts of a call where somebody speaks. Silence costs compute time, and in long silences Whisper sometimes invents text. **Voice activity detection** (VAD) finds the speech.

The simplest VAD cuts the audio into 30 ms **frames**, measures the loudness of each frame, and calls a frame speech when it is louder than a **threshold**. Two settings make the result usable:

- **hangover**: a pause shorter than this (0.3 s) stays inside the speech, so one sentence is one region, not ten.
- **minimum speech**: a region shorter than 0.1 s (a click) is dropped.

Run the next cell. It defines the energy VAD.

In [ ]:
FRAME = 0.03  # 30 ms frames

def frame_db(x, sr):
    """The loudness of each 30 ms frame, in dBFS."""
    n = int(sr * FRAME)
    frames = x[: len(x) // n * n].reshape(-1, n)
    return 20 * np.log10(np.sqrt(np.mean(frames ** 2, axis=1)) + 1e-9)

def regions(flags, hangover=0.3, min_speech=0.1):
    """Speech frames -> (start, end) times. A pause shorter than `hangover` seconds stays inside the speech."""
    out, start, quiet = [], None, 0
    keep = int(round(hangover / FRAME))
    for i, f in enumerate(flags):
        if f:
            start = i if start is None else start
            quiet = 0
        elif start is not None:
            quiet += 1
            if quiet > keep:
                end = i - quiet + 1
                if (end - start) * FRAME >= min_speech:
                    out.append((start * FRAME, end * FRAME))
                start, quiet = None, 0
    if start is not None:
        out.append((start * FRAME, (len(flags) - quiet) * FRAME))
    return out

def energy_vad(x, sr, threshold_db=-40.0, hangover=0.3):
    """The baseline: a frame is speech when it is louder than threshold_db."""
    return regions(frame_db(x, sr) > threshold_db, hangover)

Now a tiny example that you can check by eye: 1 second of quiet hiss, 1 second of a humming "voice", 1 second of hiss.

> **Predict.** At a threshold of -40 dBFS, which times will the VAD call speech?

In [ ]:
sr = 8000
rng = np.random.default_rng(0)
t = np.arange(sr) / sr
quiet = 0.001 * rng.standard_normal(sr)                       # 1 s of line hiss
voice = 0.1 * np.sin(2 * np.pi * 220 * t) * (1 + np.sin(2 * np.pi * 3 * t)) / 2   # 1 s of a humming "voice"
tiny = np.concatenate([quiet, voice, quiet]).astype(np.float32)
db = frame_db(tiny, sr)
print("frame loudness (dBFS), every 5th frame:", np.round(db[::5]).astype(int))
print("speech regions at -40 dBFS:", energy_vad(tiny, sr, -40))

> **Check.** The hiss is near -60 dBFS and the hum is near -25 dBFS, so you should see one region from about 1.0 to 2.0 seconds. The hum fades in and out, so the edges move a little.

## 2. Score it against the true turns

Our calls have exact labels: the start and end of every word. Two numbers score a VAD, frame by frame (every 10 ms):

- **missed speech**: the share of true speech that the VAD did not find. Missed speech is lost for the recogniser.
- **false speech**: the share of non-speech that the VAD called speech. False speech costs compute, and can become invented text.

In [ ]:
def frame_scores(pred, truth, seconds, step=0.01):
    """missed = share of true speech not found; false = share of non-speech called speech."""
    n = int(seconds / step) + 1
    def mask(spans):
        m = np.zeros(n, bool)
        for s, e in spans:
            m[int(s / step):int(e / step)] = True
        return m
    p, t = mask(pred), mask(truth)
    return {"missed": float((t & ~p).sum() / max(t.sum(), 1)), "false": float((p & ~t).sum() / max((~t).sum(), 1))}

Call C-0022 is a damaged lantern. The agent puts the caller on hold, with music, for about 17.7 seconds: a practice problem we made on purpose. Run the next cell.

In [ ]:
calls = load_calls()
call = calls["C-0022"]
x, sr = read_audio("data/calls/phone/C-0022.wav")
truth = [(s, e) for s, e, _ in true_turns(call)]
pred = energy_vad(x, sr, threshold_db=-40)
score = frame_scores(pred, truth, call["duration"])
print(f"{len(pred)} speech regions; missed speech {score['missed']:.3f}, false speech {score['false']:.3f}")

> **Check.** You should see 22 regions, missed speech 0.19 and false speech 0.626.

The next cell draws the loudness of each frame, the threshold, the true turns (green) and the VAD's regions (orange).

In [ ]:
db = frame_db(x, sr)
fig, ax = plt.subplots(figsize=(12, 3.4))
ax.plot(np.arange(len(db)) * FRAME, db, lw=0.5, color="grey")
ax.axhline(-40, color="black", ls="--", lw=1, label="threshold -40 dBFS")
for s, e in truth:
    ax.axvspan(s, e, ymin=0.92, ymax=1.0, color="green", alpha=0.6)
for s, e in pred:
    ax.axvspan(s, e, ymin=0.84, ymax=0.91, color="orange", alpha=0.8)
ax.set_xlabel("seconds"); ax.set_ylabel("dBFS"); ax.set_ylim(-90, 0); ax.legend(loc="lower right")
ax.set_title("C-0022: true turns (green, top) and energy VAD (orange)"); plt.show()

## 3. Predict: what happens during the music?

> **Predict.** Music on hold is not speech. Is it louder or quieter than -40 dBFS? How much of the hold will the VAD call speech?

In [ ]:
hold = [t for t in call["turns"] if t.get("hold")][0]
hold_start, hold_end = hold["start"] - hold["silence_before"], hold["start"]
in_hold = sum(max(0, min(e, hold_end) - max(s, hold_start)) for s, e in pred)
print(f"music on hold from {hold_start:.1f} to {hold_end:.1f} s: {in_hold:.1f} s of it called speech")

> **Check.** About 17.1 seconds of the 17.7-second hold are called speech. Energy does not know what speech is: music is loud, so it passes. A recogniser then gets music, and may write words for it.

## 4. The noise set

The noise set has 20 seconds of each kind of sound that is not speech. Run the next cell: it shows how much of each the energy VAD calls speech.

In [ ]:
for name in ["music_on_hold", "background_talk", "street", "line"]:
    noise, nsr = read_audio(f"data/noise/{name}.wav")
    found = energy_vad(noise, nsr, threshold_db=-40)
    share = sum(e - s for s, e in found) / (len(noise) / nsr)
    print(f"{name:16} loudness {dbfs(noise):6.1f} dBFS   called speech: {share:.2f} of 20 s")

> **Check.** Music, background talk and street noise are called speech for the whole 20 seconds (1.0, 1.0 and 1.0 of the time). Only the quiet line hiss is not (0.0). The next lesson compares two VADs that look at the shape of the sound, not only its loudness.

## 5. Change one thing: the hangover

> **Predict.** With no hangover (0 s), are there more regions or fewer? Does missed speech go up or down?

In [ ]:
for hangover in (0.0, 0.3, 1.0):
    p = energy_vad(x, sr, threshold_db=-40, hangover=hangover)
    s = frame_scores(p, truth, call["duration"])
    print(f"hangover {hangover:.1f} s: {len(p):3} regions, missed {s['missed']:.3f}, false {s['false']:.3f}")

> **Check.** With no hangover, the short pauses between words cut the speech into many small regions, and the quiet starts and ends of words are missed. A long hangover joins everything, and false speech goes up.

## 6. Failure case: a threshold in decibels depends on the line

A threshold that works on one line fails on a quieter line. The next cell plays the same call 20 dB quieter (as a weaker phone line or a lower gain would).

In [ ]:
softer = x * 0.1   # the same call, 20 dB quieter (another phone line, another gain)
s = frame_scores(energy_vad(softer, sr, threshold_db=-40), truth, call["duration"])
print(f"20 dB quieter: missed {s['missed']:.3f}, false {s['false']:.3f}")

> **Check.** Missed speech goes from 0.19 to 0.754: most of the speech is now under -40 dBFS. **Fix:** set the threshold from the call itself (for example a number of decibels above the quietest 10% of frames), or use a VAD that does not depend on loudness.

## 7. Your turn: pick a threshold

> **Your turn.** For each threshold in `[-55, -50, -45, -40, -35, -30, -25]`, run `energy_vad` (hangover 0.3 s) on the six validation calls in `task_calls` (both channels mixed, as `read_audio` does), score each call with `frame_scores` against its true turns, and compute **mean missed + mean false** over the six calls. Put the totals in a dictionary `totals` (threshold → total), and the threshold with the lowest total in `best_threshold`.

In [ ]:
thresholds = [-55, -50, -45, -40, -35, -30, -25]
task_calls = ["C-0022", "C-0098", "C-0204", "C-0093", "C-0181", "C-0003"]
totals = {}
for threshold in thresholds:
    missed, false = [], []
    for call_id in task_calls:
        c = calls[call_id]
        audio, rate = read_audio(f"data/calls/phone/{call_id}.wav")
        s = frame_scores(energy_vad(audio, rate, threshold_db=threshold),
                         [(a, b) for a, b, _ in true_turns(c)], c["duration"])
        missed.append(s["missed"]); false.append(s["false"])
    totals[threshold] = float(np.mean(missed) + np.mean(false))
    print(threshold, round(np.mean(missed), 3), round(np.mean(false), 3), round(totals[threshold], 3))
best_threshold = min(totals, key=totals.get)
print("best:", best_threshold)

In [ ]:
expect('best_threshold', best_threshold, -30)

In [ ]:
expect("the lowest total", None if best_threshold is ... else round(totals[best_threshold], 3), 0.473, tolerance=0.002)

> **Check.** Even the best threshold leaves a lot of missed or false speech on these calls. Adding the two numbers treats them as equally bad. Is that right for Larkfield? Missed speech loses words; false speech costs compute and invented text.

## Next

You built an energy VAD, scored it against the true turns, and saw it fail on music and noise. The [lesson page](https://learning.nextia-ai.com/courses/speech/m03/speech-or-silence/) has the diagrams and the knowledge checks that count toward your certificate.

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

**Next lesson:** [WebRTC VAD and Silero VAD](https://learning.nextia-ai.com/courses/speech/m03/webrtc-vad-and-silero-vad/) · **This lesson:** [read it on the site](https://learning.nextia-ai.com/courses/speech/m03/speech-or-silence/)

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/notebook-footer-light.png" alt="Nextia Learning · learning.nextia-ai.com" width="100%">

Free, practical AI courses at [learning.nextia-ai.com](https://learning.nextia-ai.com). Every call is synthetic: open text-to-speech voices read scripts that we invented. No real customer and no real person's voice is in this data.